# 03 - Evaluation
Extraction quality vs CUAD v1 expert labels and LLM-judged groundedness of risk explanations. The eval itself runs as the bundle job `eval_job` (serverless); this notebook reads the logged results from Unity Catalog and the MLflow run.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
sql("SELECT clause_type, basis, count(*) docs, count_if(gt_present) gt_present FROM ${catalog}.ml.eval_ground_truth GROUP BY ALL ORDER BY 1")

,clause_type,basis,docs,gt_present
0,assignment_restriction,yes_no,99,72
1,auto_renewal,span_proxy,99,29
2,effective_date,span,99,76
3,exclusivity,yes_no,99,37
4,expiration_date,span,99,81
5,governing_law,span,99,85
6,liability_cap,yes_no,99,47
7,non_compete,yes_no,99,25
8,notice_period_days,span,99,17
9,parties,span,99,99


In [3]:
from export_evidence import latest_eval_run
run_id, exp_id, ts = latest_eval_run(); print('MLflow experiment', exp_id, '| run', run_id, '| at', ts)
run = ws().experiments.get_run(run_id).run
print(run.info.run_name, run.info.status)
pd.DataFrame([(m.key, m.value) for m in run.data.metrics if '.' not in m.key], columns=['metric','value']).sort_values('metric')

MLflow experiment 702680364159558 | run c3d5551824c7460e8f811c5691c0154e | at 2026-11-18T02:56:23.000Z
eval-ai_extract:v2.1-20261118-024345 RunInfoStatus.FINISHED


,metric,value
0,cost_per_1k_docs_usd,108.640000
1,explanation_groundedness/mean,0.911765
2,groundedness,0.911765
3,judge_failures,0.000000
4,judged_explanations,136.000000
5,key_clause_f1,0.764356
6,key_clause_precision,0.869369
7,key_clause_recall,0.681979
8,macro_f1,0.768894
9,micro_f1,0.827479


In [4]:
sql(f"SELECT clause_type, tp, fp, fn, round(precision,3) p, round(recall,3) r, round(f1,3) f1, value_compared, round(exact_match_rate,3) exact, round(fuzzy_match_rate,3) fuzzy FROM {CATALOG}.ml.eval_results WHERE mlflow_run_id = '{run_id}' AND clause_type <> '__summary__' ORDER BY f1")

,clause_type,tp,fp,fn,p,r,f1,value_compared,exact,fuzzy
0,uncapped_liability,13,13,11,0.5,0.542,0.52,0,NaN,NaN
1,auto_renewal,14,2,15,0.875,0.483,0.622,0,NaN,NaN
2,termination_for_convenience,23,13,11,0.639,0.676,0.657,0,NaN,NaN
3,expiration_date,41,2,40,0.953,0.506,0.661,38,0.974,0.974
4,notice_period_days,14,10,3,0.583,0.824,0.683,14,1.0,1.0
5,non_compete,22,13,3,0.629,0.88,0.733,0,NaN,NaN
6,renewal_term,21,6,8,0.778,0.724,0.75,21,0.0,0.619
7,exclusivity,32,14,5,0.696,0.865,0.771,0,NaN,NaN
8,liability_cap,35,1,12,0.972,0.745,0.843,0,NaN,NaN
9,effective_date,72,16,4,0.818,0.947,0.878,66,0.955,0.955


In [5]:
sql(f"SELECT metric_name, round(metric_value,4) value FROM {CATALOG}.ml.eval_results WHERE mlflow_run_id = '{run_id}' AND clause_type = '__summary__' ORDER BY 1")

,metric_name,value
0,cost_per_1k_docs_usd,108.64
1,groundedness,0.9118
2,judge_failures,0.0
3,judge_retried_ok,0.0
4,judged_explanations,136.0
5,key_clause_f1,0.7644
6,key_clause_precision,0.8694
7,key_clause_recall,0.682
8,macro_f1,0.7689
9,micro_f1,0.8275


In [6]:
sql("SELECT clause_type, gt_answer, predicted_value, exact_match, fuzzy_match FROM ${catalog}.ml.eval_details WHERE clause_type IN ('governing_law','expiration_date','notice_period_days') AND exact_match IS NOT NULL ORDER BY clause_type LIMIT 15")

,clause_type,gt_answer,predicted_value,exact_match,fuzzy_match
0,expiration_date,12/31/20,2020-12-31,true,true
1,expiration_date,3/20/05,2005-03-20,true,true
2,expiration_date,11/14/18,2018-11-14,true,true
3,expiration_date,10/31/17,2017-10-31,true,true
4,expiration_date,9/10/16,2016-09-10,true,true
5,expiration_date,1/1/36,2036-01-01,true,true
6,expiration_date,5/31/16,2016-05-31,true,true
7,expiration_date,12/31/20,2020-12-31,true,true
8,expiration_date,3/1/13,2013-03-01,true,true
9,expiration_date,1/18/22,2022-01-18,true,true


In [7]:
sql("SELECT grounded, count(*) n FROM ${catalog}.ml.eval_groundedness GROUP BY 1")

,grounded,n
0,true,124
1,false,12


In [8]:
sql("SELECT g.rule_id, left(g.explanation, 200) explanation, left(e.rationale, 220) rationale FROM ${catalog}.ml.eval_groundedness e JOIN ${catalog}.curated.gold_risk_flags g USING (flag_id) WHERE NOT e.grounded LIMIT 5")

,rule_id,explanation,rationale
0,R02,"This clause triggers the rule because it explicitly restricts the Talent party from competing by providing that ""Tal...",The explanation contains a factual error about the contract. While the first part correctly identifies that the clau...
1,R02,"This clause triggers the rule because i-Escrow is restricted from running ""banner advertisements on the Co-Branded S...",The explanation contains a factual inaccuracy about the contract. The explanation states that the restriction on run...
2,R02,"This clause triggers the rule because Theismann ""shall not use, permit the use of, or license to others the Property...",The explanation makes a factual assertion about the contract that is not supported by the source snippet. Specifical...
3,R05,"Not supported by the cited text - needs reviewer check.\n\nThe excerpt explicitly states the arrangement is ""non-exc...",The explanation makes an unsupported factual assertion about the contract. While the explanation correctly identifie...
4,R04,"This clause triggers the rule because it requires the parties to agree on renewal terms ""Ninety (90) days prior to t...","The explanation mischaracterizes what the contract actually says. The source snippet states that ""Ninety (90) days p..."


In [9]:
sql("SELECT stage, docs, pages, input_tokens, output_tokens, est_cost_usd, cost_per_1k_docs FROM ${catalog}.curated.gold_cost_metrics ORDER BY stage")

,stage,docs,pages,input_tokens,output_tokens,est_cost_usd,cost_per_1k_docs
0,classify,100,NaN,429146,3000,0.2191,2.19
1,explain,63,NaN,65915,13282,0.1323,1.32
2,extract,100,NaN,966279,66425,0.5828,5.83
3,parse,100,1986,NaN,NaN,9.93,99.3
4,total,100,1986,1461340,82707,10.8642,108.64
